# AICIDENTIA - 03 RQ1: risk themes and trends

**RQ1:** What latent risk themes exist in reported AI incidents, and which have grown significantly over time? Compare LDA vs NMF vs BERTopic; test trends with Mann-Kendall.

> CRISP-DM phases: **Modeling** (topic models) and **Evaluation** (coherence, diversity, stability, trend tests)

This notebook covers **RQ1a (themes, Phase 3)** and **RQ1b (trends, Phase 4)**. Logic lives in `aicidentia.features.text`, `aicidentia.models.topics` and `aicidentia.models.topic_experiment` and `aicidentia.models.trend_outputs`; the write-ups are `docs/rq1_topic_models.md` and `docs/rq1_trends.md`. Numbers come from the results registry (`rq1a.*`) or the tables.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
pd.set_option("display.width", 200)
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. One shared corpus for every model

All three families see the same tokens (lower-case, standard and domain stopwords removed, WordNet lemmas, document-frequency filter), so their coherence scores are comparable. BERTopic additionally uses sentence embeddings of the untouched text.

In [ ]:
from aicidentia.data import clean
from aicidentia.features.text import build_corpus

incidents = clean.load_processed("incidents_clean")
corpus = build_corpus(incidents["incident_id"], incidents["text"])
print(
    len(corpus.docs),
    "incidents |",
    len(corpus.vocabulary),
    "words |",
    round(sum(map(len, corpus.tokens)) / len(corpus.tokens), 1),
    "tokens/incident",
)
print(corpus.texts[0])
print(corpus.tokens[0])

## 2. Fit and score the grid

26 configurations x 5 seeds. Every fit is cached in `data/interim/` under a hash of the corpus, parameters and seed, so this cell takes seconds after the first run (the first full run took several minutes on CPU).

In [ ]:
from aicidentia.models import topic_experiment as E

result = E.run_experiment(corpus)
display(
    result.summary[
        [
            "label",
            "n_topics_mean",
            "c_v_mean",
            "c_v_sd",
            "c_npmi_mean",
            "c_npmi_sd",
            "diversity_mean",
            "stability",
            "outlier_share_before_mean",
            "eligible",
        ]
    ].round(3)
)

## 3. Choose a model by a rule fixed in advance

Eligible = 8-25 topics. Best configuration per contender by summed rank of C_v and NPMI; winner by mean rank over C_v, NPMI, diversity and stability. The rule is re-run under four other weightings as a sensitivity check.

In [ ]:
selection = E.select_model(result.summary)
display(
    selection.best_per_contender[
        [
            "contender",
            "label",
            "n_topics_mean",
            "c_v_mean",
            "c_npmi_mean",
            "diversity_mean",
            "stability",
        ]
    ].round(3)
)
display(selection.ranks.round(2))
print("Winner:", selection.winner_config)
print(pd.Series(selection.sensitivity, name="winner under rule").to_string())

In [ ]:
display(Image(filename=FIG / "fig_rq1_1_coherence_by_config.png"))
display(Image(filename=FIG / "fig_rq1_2_model_comparison.png"))

**Reading the result.** The advanced model does not win here: NMF on TF-IDF beats both BERTopic variants on coherence, while BERTopic clearly beats LDA. That is reported as found. Coherence is a proxy, not a measurement of interpretability: your label review (below) is the real test.

## 4. Chosen model: topics and your label review

`write_all` saves per-incident assignments to `data/processed/incident_topics.parquet`, writes the tables, figures and `docs/rq1_topic_models.md`, and creates `data/manual/topic_labels.csv` with a **blank `final_label`** for you (an existing file with your answers is never overwritten).

In [ ]:
from aicidentia.models import topic_outputs

out = topic_outputs.write_all(corpus, result, selection)
print("topic_labels.csv:", out["label_status"], "| model_key:", out["model_key"])
display(out["table"][["topic_id", "n_incidents", "suggested_label", "top_10_terms"]])
display(Image(filename=FIG / "fig_rq1_3_topic_sizes.png"))

In [ ]:
labels = pd.read_csv(
    PROJECT_ROOT / "data" / "manual" / "topic_labels.csv", dtype=str, keep_default_na=False
)
print(
    "final_label answered:", int((labels["final_label"].str.strip() != "").sum()), "of", len(labels)
)
display(
    labels[["topic_id", "suggested_label", "final_label", "representative_titles"]].assign(
        representative_titles=lambda d: (
            d["representative_titles"].str.split(" \\|\\| ").str[:2].str.join(" | ")
        )
    )
)

## 5. Registry check

In [ ]:
for key in [
    "rq1a.selection.winner_config",
    "rq1a.chosen.n_topics",
    "rq1a.chosen.c_v_mean",
    "rq1a.chosen.stability",
    "rq1a.chosen.dominant_weight_median",
    "rq1a.manual.topic_labels_status",
]:
    print(key, "=", registry.get(key))

## 6. Trends over time (RQ1b, Phase 4)

Themes come from the model above and **your labels** in `data/manual/topic_labels.csv`. While labels are blank, themes carry placeholder names and every result is marked PROVISIONAL; giving two topics the same `final_label` merges them. Only complete years in the Phase 1 window are tested; the partial year is shown, never tested.

In [ ]:
from aicidentia.models import trend_outputs

trend = trend_outputs.run_trends()
trend_outputs.write_all(trend)
f = trend.facts
print(
    "provisional:", f["provisional"], "| window:", f["window"], "| partial year:", f["partial_year"]
)
print("emerging themes:", f["emerging_themes"])

**Mann-Kendall with Sen's slope, Benjamini-Hochberg-adjusted, on yearly shares (the primary measure).** *Emerging* = significantly increasing AND the last-two-year share above the long-run mean.

In [ ]:
t = trend.tests["themes"]["share"]
display(
    t[
        [
            "group",
            "tau",
            "sen_slope",
            "p",
            "p_bh",
            "p_hamed_rao",
            "longrun_mean",
            "recent_mean",
            "direction",
            "emerging",
        ]
    ].round(4)
)
display(Image(filename=FIG / "fig_rq1b_1_theme_heatmap.png"))
display(Image(filename=FIG / "fig_rq1b_2_emerging_themes.png"))

**Reporting-bias sensitivity.** Counts mix real change with database growth; shares remove growth; report-weighted shares add media attention (but recent incidents have had less time to attract reports). `agree` compares those three; `window_sensitive` flags a change in the shorter window, which mostly reflects lower power.

In [ ]:
display(trend.classification["themes"])
print(
    {
        k: f[k]
        for k in [
            "increasing_counts",
            "increasing_share",
            "increasing_wshare",
            "n_themes_disagree",
            "n_themes_window_sensitive",
        ]
    }
)
display(trend.tables["reports_by_year"].round(2))
display(Image(filename=FIG / "fig_rq1b_3_trend_by_measure.png"))

**MIT risk domains** (shares among MIT-labelled incidents; label coverage falls for recent years, so a trend can partly reflect which incidents were labelled).

In [ ]:
d = trend.tests["domains"]["share"]
display(d[["group", "tau", "sen_slope", "p", "p_bh", "direction", "emerging"]].round(4))
display(trend.tables["mit_coverage"].round(3))

**Forecasting evaluation** (quarterly counts, rolling origin, MASE). Reported honestly: the question is whether forecasting is justified, not just which model is least bad.

In [ ]:
display(trend.forecast_summary.round(3))
print(
    "justified:",
    f["forecast_justified"],
    "| best:",
    f["forecast_best_model"],
    round(f["forecast_best_mase"], 2),
    "vs naive",
    round(f["forecast_naive_mase"], 2),
)
print("not scored:", trend.forecast_skipped.to_dict("records"))
display(Image(filename=FIG / "fig_rq1b_4_forecast_mase.png"))

**OECD AIM replication** is not performed unless an export is present in `data/external/`.

In [ ]:
print("files in data/external:", f["oecd_aim_files"])
for key in ["rq1b.emerging_themes", "rq1b.forecast_justified", "rq1b.provisional"]:
    print(key, "=", registry.get(key))